## Section 1: Load and Inspect Sentinel-2 Data (8 min)

### Import Libraries

In [ ]:
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import os
from tqdm import tqdm


print("✅ Libraries imported successfully")

✅ Libraries imported successfully


### Define Data Paths

In [ ]:
# Set paths
project_dir = Path(os.getenv('SCRATCH_training2600')) / Path(os.getenv('USER'))
data_dir = project_dir / 'data' / 'sentinel2'
output_dir = project_dir / 'data' / 'preprocessed'
output_dir.mkdir(parents=True, exist_ok=True)


# Metadata produced in Lab 3
tile_metadata_path = data_dir / 'tile_metadata.csv'
if not tile_metadata_path.exists():
    raise FileNotFoundError(f"tile_metadata.csv not found at {tile_metadata_path}. Run Lab 3 exports first.")


tile_metadata = pd.read_csv(tile_metadata_path)
mgrs_tile = tile_metadata['mgrs_tile'].iloc[0]
corine_path = Path(tile_metadata['corine_label'].iloc[0])
if not corine_path.is_absolute():
    corine_path = data_dir / corine_path


# Sentinel-2 acquisitions (exclude CORINE label)
scene_files = sorted([p for p in data_dir.glob('*.tif') if 'corine' not in p.name.lower()])


print(f"Data directory: {data_dir}")
print(f"Output directory: {output_dir}")
print(f"MGRS tile: {mgrs_tile}")
print(f"CORINE label: {corine_path}")
print(f"\n✅ Found {len(scene_files)} Sentinel-2 scenes:")
for f in scene_files:
    print(f"   - {f.name}")

Data directory: /p/scratch/training2600/hashim1/data/sentinel2
Output directory: /p/scratch/training2600/hashim1/data/preprocessed

✅ Found 4 Sentinel-2 scenes:
   - 20240629T125309_20240629T125303_T27WVM.tif
   - 20240707T130301_20240707T130301_T27WWM.tif
   - 20240913T131259_20240913T131255_T27WWM.tif
   - 20240915T130301_20240915T130257_T27WVM.tif


### Load and Inspect a Scene

In [ ]:
# Inspect first scene and derive band indices
scene_path = scene_files[0]


# Desired Sentinel-2 bands (10 m + 20 m resampled by GEE)
target_bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']


with rasterio.open(scene_path) as src:
    descriptions = [d if d is not None else '' for d in src.descriptions]
    band_lookup = {name: idx + 1 for idx, name in enumerate(descriptions)}
    missing = [b for b in target_bands if b not in band_lookup]
    if missing:
        raise ValueError(f"Missing expected bands {missing} in {scene_path.name}. Check Lab 3 export bands.")
    band_indices = [band_lookup[b] for b in target_bands]
    band_names = target_bands
    data = src.read(indexes=band_indices)  # Shape: (bands, height, width)
    profile = src.profile


print("📊 Scene Metadata:")
print(f"   File: {scene_path.name}")
print(f"   Dimensions: {data.shape[2]} x {data.shape[1]} pixels")
print(f"   Selected bands: {band_names}")
print(f"   CRS: {profile['crs']}")
print(f"   Resolution: {profile['transform'][0]} meters")
print(f"   Data Type: {profile['dtype']}")


print(f"\n✅ Loaded data shape: {data.shape}")
print(f"   Data range: [{data.min()}, {data.max()}]")
print(f"   Data type: {data.dtype}")

📊 Scene Metadata:
   Dimensions: 337 x 985 pixels
   Bands: 6
   CRS: EPSG:32627
   Resolution: (80.0, 80.0) meters
   Bounds: BoundingBox(left=482840.0, bottom=7090160.0, right=509800.0, top=7168960.0)
   Data Type: uint16

✅ Loaded data shape: (6, 985, 337)
   Data range: [0, 9370]
   Data type: uint16


In [ ]:
# Load CORINE label and align to the Sentinel-2 grid
with rasterio.open(corine_path) as lbl_src:
    corine_nodata = lbl_src.nodata if lbl_src.nodata is not None else 0
    if (lbl_src.height == profile['height'] and
        lbl_src.width == profile['width'] and
        lbl_src.transform == profile['transform']):
        corine_aligned = lbl_src.read(1)
    else:
        corine_aligned = np.zeros((profile['height'], profile['width']), dtype=lbl_src.dtypes[0])
        reproject(
            source=rasterio.band(lbl_src, 1),
            destination=corine_aligned,
            src_transform=lbl_src.transform,
            src_crs=lbl_src.crs,
            dst_transform=profile['transform'],
            dst_crs=profile['crs'],
            resampling=Resampling.nearest
        )


unique_codes, unique_counts = np.unique(corine_aligned, return_counts=True)
print(f"CORINE codes present (top 10): {list(zip(unique_codes, unique_counts))[:10]}")
print(f"Label nodata value: {corine_nodata}")

### Visualize RGB Composite

In [59]:
# Assuming bands: B2, B3, B4, B8, B11, B12
# RGB = B4, B3, B2 (Red, Green, Blue)
band_names = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
rgb_indices = [2, 1, 0]  # B4=2, B3=1, B2=0 (0-indexed)

# Extract RGB
rgb = data[rgb_indices, :, :].transpose(1, 2, 0)  # (H, W, 3)

# Normalize to 0-1 for display (clip at 2nd and 98th percentile)
def normalize_for_display(img, percentile=2):
    vmin, vmax = np.percentile(img, [percentile, 100-percentile])
    img_norm = np.clip((img - vmin) / (vmax - vmin), 0, 1)
    return img_norm

rgb_display = normalize_for_display(rgb)

# Plot
plt.figure(figsize=(12, 8))
plt.imshow(rgb_display)
plt.title(f"RGB Composite: {scene_path.name}", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"\n📈 Statistics per band:")
for i, name in enumerate(band_names):
    band_data = data[i]
    print(f"   {name}: mean={band_data.mean():.2f}, std={band_data.std():.2f}, range=[{band_data.min()}, {band_data.max()}]")


📈 Statistics per band:
   B2: mean=319.06, std=377.18, range=[0, 9370]
   B3: mean=397.70, std=449.30, range=[0, 9342]
   B4: mean=393.91, std=462.08, range=[0, 9248]
   B8: mean=1100.26, std=1292.56, range=[0, 8085]
   B11: mean=1062.13, std=1199.01, range=[0, 5568]
   B12: mean=641.03, std=708.39, range=[0, 5076]


## Section 2: Patch Extraction (10 min)

### Why Patches?
Deep learning models work on fixed-size inputs. We divide large satellite images into smaller patches:
- **Patch Size:** 224×224 or 256×256 pixels (common for vision models)
- **Overlap:** Optional overlap between patches for better coverage
- **Stride:** Controls spacing between patches

### Extract Patches Function

In [ ]:
# CORINE → simplified class mapping (keep it small for the course)
CLASS_GROUPS = {
    'Urban': [111, 112, 121, 122, 123, 124, 131, 132, 133, 141, 142],
    'Agriculture': [211, 212, 213, 221, 222, 223, 231, 241, 242, 243, 244],
    'Forest': [311, 312, 313],
    'Shrubland': [321, 322, 323, 324],
    'Bare': [331, 332, 333, 334, 335],
    'Wetland': [411, 412, 421, 422, 423],
    'Water': [511, 512, 513, 521, 522, 523],
}


code_to_class = {}
for idx, (_, codes) in enumerate(CLASS_GROUPS.items()):
    for code in codes:
        code_to_class[code] = idx


def remap_corine(label_array, nodata_value=0):
    mapped = np.full(label_array.shape, -1, dtype=np.int16)
    for code, cls_idx in code_to_class.items():
        mapped[label_array == code] = cls_idx
    mapped[label_array == nodata_value] = -1
    return mapped


corine_classes = remap_corine(corine_aligned, nodata_value=corine_nodata)


def extract_patches_with_labels(image, label, patch_size=224, stride=224, min_valid_pixels=0.8, min_label_pixels=0.8):
    """Extract patches where both imagery and labels are sufficiently valid."""
    bands, height, width = image.shape
    patches = []
    labels = []
    positions = []
    

    for i in range(0, height - patch_size + 1, stride):
        for j in range(0, width - patch_size + 1, stride):
            patch_img = image[:, i:i+patch_size, j:j+patch_size]
            patch_lbl = label[i:i+patch_size, j:j+patch_size]
            
            valid_img_ratio = (patch_img != 0).sum() / patch_img.size
            valid_lbl_ratio = (patch_lbl != -1).sum() / patch_lbl.size
            
            if valid_img_ratio < min_valid_pixels or valid_lbl_ratio < min_label_pixels:
                continue
            
            lbl_vals = patch_lbl[patch_lbl != -1].flatten()
            majority_label = np.bincount(lbl_vals, minlength=len(CLASS_GROUPS)).argmax()
            
            patches.append(patch_img)
            labels.append(majority_label)
            positions.append((i, j))
    
    return patches, labels, positions


print("✅ Patch extraction with CORINE labels ready")

✅ Patch extraction function defined


In [ ]:
# Extract patches from all scenes for this tile
patch_size = 224
stride = 224  # No overlap (stride = patch_size)


all_patches = []
all_labels = []
all_positions = []
scene_names = []


print(f"🔪 Extracting {patch_size}x{patch_size} patches with stride {stride} from {len(scene_files)} scenes...")


for scene_path in scene_files:
    with rasterio.open(scene_path) as src:
        image = src.read(indexes=band_indices)
    patches, labels_int, positions = extract_patches_with_labels(
        image, corine_classes, patch_size=patch_size, stride=stride, min_valid_pixels=0.8, min_label_pixels=0.8
    )
    all_patches.extend(patches)
    all_labels.extend(labels_int)
    all_positions.extend([(scene_path.name, pos) for pos in positions])
    scene_names.append(scene_path.name)
    print(f"   {scene_path.name}: {len(patches)} patches")


patches = np.array(all_patches)
labels = np.array(all_labels)


print(f"\n✅ Extracted {len(patches)} valid patches across {len(scene_files)} scenes")
print(f"   Patch shape: {patches[0].shape}")
print(f"   Label distribution: {pd.Series(labels).value_counts().sort_index().to_dict()}")

🔪 Extracting 224x224 patches with stride 224...

✅ Extracted 0 valid patches


IndexError: list index out of range

### Visualize Sample Patches

In [ ]:
# Display first 6 patches
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()


for idx in range(min(6, len(patches))):
    patch = patches[idx]
    rgb_patch = patch[[2, 1, 0], :, :].transpose(1, 2, 0)  # B4, B3, B2
    rgb_patch_display = normalize_for_display(rgb_patch)


    axes[idx].imshow(rgb_patch_display)
    axes[idx].set_title(f"Patch {idx+1} | Scene: {all_positions[idx][0]} | Pos: {all_positions[idx][1]}", fontsize=9)
    axes[idx].axis('off')


plt.tight_layout()
plt.show()
print("done")

done


## Section 3: Normalization Techniques (8 min)

### Why Normalize?
Neural networks train better with normalized inputs:
- **Faster convergence:** Reduces gradient magnitude variation
- **Numerical stability:** Avoids overflow/underflow
- **Better generalization:** Removes scaling bias

### Common Normalization Methods
1. **Min-Max Scaling:** Scale to [0, 1]
2. **Standardization (Z-score):** Mean=0, Std=1
3. **Percentile Clipping:** Robust to outliers

In [70]:
def normalize_minmax(patches, data_min=None, data_max=None):
    """
    Min-Max normalization to [0, 1].
    """
    patches_arr = np.array(patches)  # (N, C, H, W)
    
    if data_min is None:
        data_min = patches_arr.min(axis=(0, 2, 3), keepdims=True)  # Per-channel min
    if data_max is None:
        data_max = patches_arr.max(axis=(0, 2, 3), keepdims=True)  # Per-channel max
    
    normalized = (patches_arr - data_min) / (data_max - data_min + 1e-8)
    return normalized, data_min, data_max


def normalize_standardize(patches, mean=None, std=None):
    """
    Standardization to mean=0, std=1.
    """
    patches_arr = np.array(patches)  # (N, C, H, W)
    
    if mean is None:
        mean = patches_arr.mean(axis=(0, 2, 3), keepdims=True)  # Per-channel mean
    if std is None:
        std = patches_arr.std(axis=(0, 2, 3), keepdims=True)  # Per-channel std
    
    normalized = (patches_arr - mean) / (std + 1e-8)
    return normalized, mean, std


def normalize_percentile(patches, lower=2, upper=98):
    """
    Percentile-based normalization (robust to outliers).
    """
    patches_arr = np.array(patches)
    
    # Compute percentiles per channel
    p_lower = np.percentile(patches_arr, lower, axis=(0, 2, 3), keepdims=True)
    p_upper = np.percentile(patches_arr, upper, axis=(0, 2, 3), keepdims=True)
    
    # Clip and normalize
    patches_clipped = np.clip(patches_arr, p_lower, p_upper)
    normalized = (patches_clipped - p_lower) / (p_upper - p_lower + 1e-8)
    
    return normalized, p_lower, p_upper

print("✅ Normalization functions defined")

✅ Normalization functions defined


In [71]:
# Apply standardization (common for pre-trained models)
print("🔄 Applying standardization...")
patches_normalized, mean, std = normalize_standardize(patches)

print(f"\n✅ Normalization complete")
print(f"   Original range: [{np.array(patches).min():.2f}, {np.array(patches).max():.2f}]")
print(f"   Normalized range: [{patches_normalized.min():.2f}, {patches_normalized.max():.2f}]")
print(f"   Normalized mean: {patches_normalized.mean():.6f}")
print(f"   Normalized std: {patches_normalized.std():.6f}")

# Save normalization parameters
norm_params = {
    'method': 'standardization',
    'mean': mean.squeeze().tolist(),
    'std': std.squeeze().tolist(),
    'band_names': band_names
}

import json
with open(output_dir / 'normalization_params.json', 'w') as f:
    json.dump(norm_params, f, indent=2)

print(f"\n💾 Saved normalization parameters to: {output_dir / 'normalization_params.json'}")

🔄 Applying standardization...

✅ Normalization complete
   Original range: [0.00, 9370.00]
   Normalized range: [-3.17, 37.61]
   Normalized mean: -0.000000
   Normalized std: 1.000000

💾 Saved normalization parameters to: /p/scratch/training2600/hashim1/data/preprocessed/normalization_params.json


## Section 4: Label Matching with CORINE Land Cover (7 min)


### About CORINE Land Cover
CORINE (Coordination of Information on the Environment) provides European land cover classification:
- **Classes:** 44 land cover types (we remap to 7 for this lab)
- **Resolution:** 100 m (already resampled to 10 m in Lab 3)
- **Updates:** Every 6 years


### Simplified Classes for ML
We remap CORINE codes to 7 classes (Urban, Agriculture, Forest, Shrubland, Bare, Wetland, Water).

In [ ]:
# Real CORINE labels (remapped)
class_names = list(CLASS_GROUPS.keys())


label_counts = pd.Series(labels).value_counts().sort_index()
print(f"✅ Label distribution (class index → count):\n{label_counts.to_dict()}")


print("\nClass index → name")
for idx, name in enumerate(class_names):
    print(f"  {idx}: {name}")


print("\nClass name → CORINE codes")
for name, codes in CLASS_GROUPS.items():
    print(f"  {name}: {codes}")

✅ Generated 1119 labels

📊 Label distribution:
Forest       776
Shrubland    163
Bare          83
Wetland       56
Water         41
Name: count, dtype: int64


In [ ]:
# Build mappings for later use
label_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_label = {idx: name for name, idx in label_to_idx.items()}


print("\n✅ Label mapping ready:")
print(label_to_idx)


✅ Label mapping:
   0: Urban (0 patches)
   1: Agricultural (0 patches)
   2: Forest (776 patches)
   3: Shrubland (163 patches)
   4: Bare (83 patches)
   5: Wetland (56 patches)
   6: Water (41 patches)


## Section 5: Train/Val/Test Split (4 min)

In [ ]:
# Split dataset: 70% train, 15% val, 15% test
from sklearn.model_selection import train_test_split


X = patches  # (N, C, H, W)
y = labels   # (N,)


# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


# Second split: 50% val, 50% test (from temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


print(f"📂 Dataset Split:")
print(f"   Train: {X_train.shape[0]} patches ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   Val:   {X_val.shape[0]} patches ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"   Test:  {X_test.shape[0]} patches ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\n   Total: {len(X)} patches")

📂 Dataset Split:
   Train: 783 patches (70.0%)
   Val:   168 patches (15.0%)
   Test:  168 patches (15.0%)

   Total: 1119 patches


## Section 6: Save Preprocessed Dataset (3 min)

In [ ]:
# Save as NumPy arrays (efficient for loading)
print("💾 Saving preprocessed dataset...\n")


np.save(output_dir / 'X_train.npy', X_train)
np.save(output_dir / 'y_train.npy', y_train)
np.save(output_dir / 'X_val.npy', X_val)
np.save(output_dir / 'y_val.npy', y_val)
np.save(output_dir / 'X_test.npy', X_test)
np.save(output_dir / 'y_test.npy', y_test)


# Save metadata
metadata = {
    'patch_size': patch_size,
    'stride': stride,
    'num_bands': len(band_names),
    'band_names': band_names,
    'num_classes': len(class_names),
    'class_names': class_names,
    'label_mapping': label_to_idx,
    'train_size': len(X_train),
    'val_size': len(X_val),
    'test_size': len(X_test),
    'normalization': norm_params,
    'mgrs_tile': mgrs_tile,
    'corine_label_path': str(corine_path)
}


with open(output_dir / 'dataset_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)


print(f"✅ Saved files to: {output_dir}")
print(f"   - X_train.npy ({X_train.nbytes / 1e6:.2f} MB)")
print(f"   - y_train.npy ({y_train.nbytes / 1e3:.2f} KB)")
print(f"   - X_val.npy ({X_val.nbytes / 1e6:.2f} MB)")
print(f"   - y_val.npy ({y_val.nbytes / 1e3:.2f} KB)")
print(f"   - X_test.npy ({X_test.nbytes / 1e6:.2f} MB)")
print(f"   - y_test.npy ({y_test.nbytes / 1e3:.2f} KB)")
print(f"   - dataset_metadata.json")
print(f"   - normalization_params.json")

💾 Saving preprocessed dataset...

✅ Saved files to: /p/scratch/training2600/hashim1/data/preprocessed
   - X_train.npy (5.41 MB)
   - y_train.npy (6.26 KB)
   - X_val.npy (1.16 MB)
   - y_val.npy (1.34 KB)
   - X_test.npy (1.16 MB)
   - y_test.npy (1.34 KB)
   - dataset_metadata.json
   - normalization_params.json


### Verify Saved Data

In [76]:
# Load and verify
X_train_loaded = np.load(output_dir / 'X_train.npy')
y_train_loaded = np.load(output_dir / 'y_train.npy')

print("🔍 Verification:")
print(f"   Loaded X_train shape: {X_train_loaded.shape}")
print(f"   Loaded y_train shape: {y_train_loaded.shape}")
print(f"   Data matches: {np.array_equal(X_train, X_train_loaded)}")
print(f"   Labels match: {np.array_equal(y_train, y_train_loaded)}")
print("\n✅ All data saved and verified successfully!")

🔍 Verification:
   Loaded X_train shape: (783, 6, 12, 12)
   Loaded y_train shape: (783,)
   Data matches: True
   Labels match: True

✅ All data saved and verified successfully!


## Summary & Next Steps


### What We Covered
✅ Loaded Sentinel-2 GeoTIFF imagery for the selected tile  
✅ Extracted 224×224 patches from all acquisitions  
✅ Applied standardization normalization  
✅ Used **real CORINE labels** remapped to 7 classes  
✅ Created train/val/test split (70/15/15)  
✅ Saved ML-ready dataset  


### Dataset Statistics
- **Total Patches:** Variable (depends on scene size and valid-label coverage)
- **Patch Size:** 224×224 pixels
- **Bands:** 6 (B2, B3, B4, B8, B11, B12)
- **Classes:** 7 remapped CORINE classes
- **Normalization:** Standardization (mean=0, std=1)
- **Format:** NumPy arrays (.npy)


### Key Preprocessing Concepts
- **Patch Extraction:** Divide large images into fixed-size inputs
- **Normalization:** Scale inputs for neural network training
- **Label Matching:** Align CORINE labels to Sentinel-2 grid and remap classes
- **Data Splitting:** Separate train/val/test to avoid overfitting


### Prepare for Lab 5.1
Next lab: **Baseline Model Training**
- We'll train a CNN classifier
- Use PyTorch
- Optionally submit training job to GPU partition
- Track training metrics


### Best Practices
1. **Always save normalization parameters** (needed for inference)
2. **Check for class imbalance** (use weighted loss if needed)
3. **Validate patches visually** (ensure no artifacts and labels align)
4. **Document metadata** (bands, resolution, classes)


### Homework (Optional)
1. Experiment with different patch sizes (128, 256, 512)
2. Try overlapping patches (stride < patch_size)
3. Compare normalization methods (min-max vs standardization)
4. Visualize class distribution per split


---


**Fantastic work!** Your data is now ready for machine learning! 🚀